インポート

In [ ]:
# もろもろインポート
import warnings
warnings.simplefilter('ignore')

# 基本のもの
import os
os.environ["OMP_NUM_THREADS"] = "2"
import numpy as np
import pandas as pd
import random


# 次元削減の関連（関数内でも参照されるため残す）
from sklearn.decomposition import PCA

from scipy import linalg

# 機械学習の関連（使用されているもののみ）
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ノートブック内でコメント扱いの optional import
# import umap

In [ ]:
from pathlib import Path
import sys

cwd = Path().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.append(str(repo_root))

In [ ]:
from src.anchor import make_anc_smote
from src.dc import get_Gfanc_dict, merge_DC
from src.models import MLP, train_and_evaluate_pytorch, evaluate_pytorch_model

In [ ]:
# データセットの取得

DATA_PATH = "data/train.csv"
data = pd.read_csv(DATA_PATH)

display(data)
# 説明変数（特徴量）をデータフレームに変換
Xall = data.iloc[:, 1:55]
display(Xall)

# 目的変数（ターゲット）をデータフレームに変換
Yall = data.iloc[:, 55]

display(Yall)
print(Yall.value_counts()) # number of each label

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np # numpyをインポートしていない場合は追加してください

# (データ全体をYallに読み込んだ直後を想定)

print("="*50)
print("ラベルを0始まりに変換します...")
print("元のラベル:", sorted(Yall.unique()))

# LabelEncoderのインスタンスを作成
le = LabelEncoder()

# Yallのデータを使って、ラベルを0から始まる数値に変換します
# (例: [1, 2, 7] -> [0, 1, 6])
# この処理は、YallがpandasのSeriesでもNumPy配列でも動作します。
Yall_encoded = le.fit_transform(Yall)

# Yallを変換後のデータで上書き
Yall = pd.Series(Yall_encoded, index=Yall.index) # YallがSeriesの場合
# Yall = Yall_encoded # YallがNumPy配列の場合

print("変換後のラベル:", sorted(np.unique(Yall)))
print("="*50)


# --- この後の処理 ---
# この修正された Yall を使って、
# クライアントへのデータ分割 (client_train_test_dataの作成など) を
# 行うようにしてください。

In [ ]:
# ===================================================================
# Step 2: 公開データ（Public Data）の抽出
# ===================================================================
print("--- Step 2: 公開データ（Public Data）の抽出 ---")

# エンコード済みの全データから、100件を公開データとして層化抽出
X_others, X_public, Y_others, Y_public = train_test_split(
    Xall, 
    Yall, 
    test_size=100,
    random_state=42,
    stratify=Yall # ラベル比率を維持
)

print(f"抽出前の全データ (Xall) のサイズ: {Xall.shape}")
print(f"公開データ (X_public) のサイズ: {X_public.shape}")
print(f"クライアント分割用の残りデータ (X_others) のサイズ: {X_others.shape}")
print("-" * 50)

In [ ]:
# ===================================================================
# Step 3: 各クライアントがk種類のラベルを持つようにデータを分割 (修正版)
# ===================================================================
print("--- Step 3: 各クライアントがk種類のラベルを持つようにデータを分割 ---")

# --- 設定 ---
NUM_CLIENTS = 100
K_LABELS_PER_CLIENT = 2 # 各クライアントが持つラベルの種類数 (k)
RANDOM_SEED = 42 # 再現性を確保するためのシード値

# シードを固定
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Y_othersがnumpy arrayの場合、pandas Seriesに変換してインデックスを扱えるようにする
if not isinstance(Y_others, pd.Series):
    Y_others = pd.Series(Y_others, index=X_others.index)

# 1. ラベルの種類と、各ラベルに対応するデータのインデックスを取得
all_labels = sorted(Y_others.unique())
indices_per_label = {label: Y_others.index[Y_others == label].tolist() for label in all_labels}

# ---【デバッグコード 1】---
# indices_per_label が正しく作成されたか確認
print("\n--- [Debug 1] indices_per_label の中身 ---")
print("目的: 各ラベルに、いくつのデータサンプルが紐づいているかを確認します。")
for label, indices in indices_per_label.items():
    print(f"  ラベル '{label}': {len(indices)} 個のサンプル")
print("-" * 30)
# ---【デバッグここまで】---

# 2. 各クライアントに k 個のラベルをランダムに割り当てる
client_label_assignments = {i: [] for i in range(NUM_CLIENTS)}
# ラベルの割り当てが偏らないように、まず全クライアントに割り当ててからシャッフルする手法も考えられますが、
# 今回は元のロジックを尊重し、各クライアントが独立してラベルを選ぶ形とします。
available_labels = list(all_labels)
for i in range(NUM_CLIENTS):
    # ラベルプールからk個を重複なくランダムに選ぶ
    # ラベルが不足しないように、毎回利用可能なラベル全体から選ぶ
    assigned_labels = np.random.choice(available_labels, K_LABELS_PER_CLIENT, replace=False)
    client_label_assignments[i] = list(assigned_labels)

# ---【デバッグコード 2】---
# client_label_assignments が正しく作成されたか確認
print("\n--- [Debug 2] client_label_assignments の中身 ---")
print("目的: 各クライアントに、どのラベルが割り当てられたか（設計図）を確認します。")
for client_id, labels in client_label_assignments.items():
    print(f"  クライアント {client_id}: ラベル {labels} を割り当て")
print("-" * 30)
# ---【デバッグここまで】---


# 3. 各ラベルがどのクライアントに割り当てられているかの逆引きマップを作成
label_to_clients_map = {label: [] for label in all_labels}
for client_id, labels in client_label_assignments.items():
    for label in labels:
        label_to_clients_map[label].append(client_id)

# ---【デバッグコード 3】---
# label_to_clients_map が正しく作成されたか確認
print("\n--- [Debug 3] label_to_clients_map の中身 ---")
print("目的: 各ラベルが、どのクライアントに所有されているか（逆引きマップ）を確認します。")
for label, client_ids in label_to_clients_map.items():
    print(f"  ラベル '{label}': クライアント {client_ids} が所有")
print("-" * 30)
# ---【デバッグここまで】---


# 4. 各ラベルのサンプルを、そのラベルを持つクライアントに均等に分割
client_indices = {i: [] for i in range(NUM_CLIENTS)}

# ラベルごとに処理を行う
for label, indices_for_this_label in indices_per_label.items():
    
    # このラベルを所有するクライアントのリストを取得
    owning_clients = label_to_clients_map.get(label, [])
    
    # このラベルを所有するクライアントがいなければ、次のラベルへ
    if not owning_clients:
        continue
    
    # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
    # ★★★ 原因修正: 元のリストを破壊しないよう、必ずコピーを作成してからシャッフルする ★★★
    # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
    shuffled_indices = indices_for_this_label.copy()
    random.shuffle(shuffled_indices)
    
    # インデックスのリストを、クライアント数に応じて分割
    index_chunks = np.array_split(shuffled_indices, len(owning_clients))
    
    # 分割したチャンクを、対応するクライアントに分配 (zipで安全にペアリング)
    for client_id, chunk in zip(owning_clients, index_chunks):
        if chunk.size > 0:
            client_indices[client_id].extend(chunk.tolist())

# ---【デバッグコード 4】---
# client_indices が正しく作成されたか確認
print("\n--- [Debug 4] client_indices の中身（データ分配直後）---")
print("目的: データ分配ロジックの結果、各クライアントにいくつのインデックスが分配されたかを確認します。")
total_assigned = 0
for client_id, indices in client_indices.items():
    print(f"  クライアント {client_id}: {len(indices)} 個のインデックスを分配")
    total_assigned += len(indices)
print(f"分配された合計インデックス数: {total_assigned} (元のデータ数: {len(Y_others)})")

# さらに詳細なチェック：client_0に分配されたインデックスの実際のラベルを調べる
print("\n--- [Debug 4 bis] client_0 の詳細チェック ---")
c0_indices = client_indices.get(0, [])
if c0_indices:
    # .locを使って、分配されたインデックスに対応するY_othersのラベルを取得
    c0_actual_labels = Y_others.loc[c0_indices].unique()
    print(f"  client_0 に割り当てられた設計図上のラベル: {client_label_assignments.get(0)}")
    print(f"  client_0 に分配されたインデックスを元データで確認した実際のラベル: {sorted(list(c0_actual_labels))}")
    # 設計図と実際のラベルが一致するかを判定
    if set(client_label_assignments.get(0, [])) == set(c0_actual_labels):
        print("  -> OK: 設計図と実際のラベルが一致しています。")
    else:
        print("  -> !!! NG: この時点で既に、設計図にないラベルのインデックスが混入しています !!!")
else:
    print("  client_0 にはデータが分配されませんでした。")
print("-" * 30)
# ---【デバッグここまで】---

# 5. client_data辞書を構築
client_data = {}
for i in range(NUM_CLIENTS):
    client_name = f"client_{i}"
    # client_indices辞書からインデックスリストを取得
    indices = client_indices.get(i, []) # .get()でキーが存在しなくてもエラーにならない
    
    if not indices:
        # このクライアントにデータが割り当てられなかった場合
        client_data[client_name] = {'X': pd.DataFrame(), 'Y': pd.Series(dtype='int')}
    else:
        X_client = X_others.loc[indices]
        Y_client = Y_others.loc[indices]
        client_data[client_name] = {'X': X_client, 'Y': Y_client}

# ▼▼▼▼▼ ここから下のコードブロックを追加 ▼▼▼▼▼
# ===================================================================
# Step 5.5: k種類のラベルを持たないクライアントを除外
# ===================================================================
print("\n--- Step 5.5: 実際のラベル数がk種類でないクライアントを除外 ---")

filtered_client_data = {}
excluded_clients_summary = []

# client_data辞書の各クライアントをチェック
for client_name, data in client_data.items():
    # 実際のデータに含まれるユニークなラベル数を取得
    num_actual_labels = data['Y'].nunique()

    # ラベル数が指定した k と一致するかを判定
    if num_actual_labels == K_LABELS_PER_CLIENT:
        # 条件を満たすクライアントは新しい辞書に追加
        filtered_client_data[client_name] = data
    else:
        # 条件を満たさないクライアントは除外リストに追加
        # client_name (e.g., "client_10") から数値IDを取得
        client_id_num = int(client_name.split('_')[1])
        assigned_labels = client_label_assignments.get(client_id_num, [])
        actual_labels = sorted(data['Y'].unique())
        reason = f"実際のラベル数({num_actual_labels})が指定のk({K_LABELS_PER_CLIENT})と不一致"
        details = f"割当: {assigned_labels}, 実際: {actual_labels}"
        excluded_clients_summary.append((client_name, reason, details))

# 元のclient_dataをフィルタリング済みのものに置き換える
client_data = filtered_client_data

print(f"フィルタリング後の最終的なクライアント数: {len(client_data)}")     
# 除外されたクライアントがいれば表示
if excluded_clients_summary:
    print("\n--- 除外されたクライアントの詳細 ---")
    for name, reason, details in excluded_clients_summary:
        print(f"クライアント '{name}': {reason} -> {details}")

# ▲▲▲▲▲ 追加するコードはここまで ▲▲▲▲▲  

# --- 結果の確認 ---
print(f"作成されたクライアントの数: {len(client_data)}")
print(f"各クライアントが持つラベルの種類数 (k): {K_LABELS_PER_CLIENT}")

# 最初の数クライアントのデータ状況を確認
for i in range(min(5, NUM_CLIENTS)):
    client_name = f"client_{i}"
    assigned_labels = client_label_assignments.get(i, [])
    actual_labels = sorted(client_data[client_name]['Y'].unique())
    
    print("-" * 20)
    print(f"クライアント '{client_name}' のデータ数: {len(client_data[client_name]['X'])}")
    print(f" 割り当てられたラベル: {assigned_labels}")
    print(f" 実際のデータラベル: {actual_labels}")
    # 詳細な分布も確認
    # print(f"クライアント '{client_name}' の実際のラベル分布:\n{client_data[client_name]['Y'].value_counts().sort_index()}")

print("-" * 50)

In [ ]:
from sklearn.model_selection import train_test_split

# ===================================================================
# Step 4: 各クライアントデータを訓練・テストデータに分割 (手動層化分割版)
# ===================================================================
print("--- Step 4: 各クライアントデータを訓練・テストデータに分割 (手動層化分割) ---")

MIN_SAMPLES_THRESHOLD = 5
TEST_SPLIT_RATIO = 0.2

client_train_test_data = {}
excluded_clients = []

for client_id, data in client_data.items():
    X_c = data['X']
    Y_c = data['Y']

    # 理由1: サンプル数がしきい値未満のクライアントは除外
    if len(X_c) < MIN_SAMPLES_THRESHOLD:
        reason = f"サンプル数が{len(X_c)}件で、しきい値({MIN_SAMPLES_THRESHOLD})未満"
        excluded_clients.append((client_id, len(X_c), reason))
        continue

    # --- ここからが手動での層化分割ロジック ---
    
    # 各クラスのサンプル数をカウント
    class_counts = Y_c.value_counts()
    
    # サンプル数が1つの希少クラスと、2つ以上の通常クラスに分ける
    rare_labels = class_counts[class_counts < 2].index
    common_labels = class_counts[class_counts >= 2].index
    
    # データを希少なものと通常なものに分割
    X_c_rare = X_c[Y_c.isin(rare_labels)]
    Y_c_rare = Y_c[Y_c.isin(rare_labels)]
    X_c_common = X_c[Y_c.isin(common_labels)]
    Y_c_common = Y_c[Y_c.isin(common_labels)]

    # 訓練データとテストデータを初期化
    # ★★★ 希少サンプルは、すべて訓練データに入れる ★★★
    X_train, Y_train = X_c_rare.copy(), Y_c_rare.copy()
    X_test, Y_test = pd.DataFrame(columns=X_c.columns), pd.Series(dtype=Y_c.dtype)

    # 通常クラスのデータが存在する場合のみ、層化分割を行う
    if not X_c_common.empty:
        # 分割後のテストサンプル数がクラス数を下回らないかチェック
        n_test_samples = max(1, int(len(Y_c_common) * TEST_SPLIT_RATIO))
        if n_test_samples < len(common_labels):
            reason = f"分割後のテストサイズ({n_test_samples})が通常クラス数({len(common_labels)})より少なく、層化抽出が不可能"
            excluded_clients.append((client_id, len(X_c), reason))
            continue
        
        # 通常データのみを層化分割
        X_train_common, X_test_common, Y_train_common, Y_test_common = train_test_split(
            X_c_common, 
            Y_c_common, 
            test_size=TEST_SPLIT_RATIO, 
            random_state=42, 
            stratify=Y_c_common
        )
        
        # 分割結果を結合
        X_train = pd.concat([X_train, X_train_common])
        Y_train = pd.concat([Y_train, Y_train_common])
        X_test = pd.concat([X_test, X_test_common])
        Y_test = pd.concat([Y_test, Y_test_common])

    # 最終的な訓練データが空になっていないか確認
    if X_train.empty:
        continue

    client_train_test_data[client_id] = {
        'X_train': X_train, 'Y_train': Y_train,
        'X_test': X_test, 'Y_test': Y_test
    }

# (以降の結果表示コードは同じ)
print(f"最終的に訓練・テスト分割されたクライアント数: {len(client_train_test_data)}")
if excluded_clients:
    print("\n--- 除外されたクライアント ---")
    for client_name, count, reason in excluded_clients:
        print(f"クライアント '{client_name}': {count} 件 -> 理由: {reason}")
print("-" * 50)
# ===================================================================
# Step 5: 結果の確認
# ===================================================================
print("--- Step 5: 結果の確認 ---")

# 例として、いずれかのクライアントのデータサイズを表示
if client_train_test_data:
    # 辞書から最初のクライアントのIDを取得
    example_client_id = next(iter(client_train_test_data))
    
    print(f"クライアント '{example_client_id}' のデータサイズ:")
    print(f"  訓練データ (X_train): {client_train_test_data[example_client_id]['X_train'].shape}")
    print(f"  テストデータ (X_test):  {client_train_test_data[example_client_id]['X_test'].shape}")

中間表現化

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ▼▼▼▼▼ 修正箇所 ▼▼▼▼▼
# --- PCA適用可能性チェック ---
print("\n--- PCA適用可能性チェック ---")
TARGET_DIM = 53 

clients_after_pca_check = {}
pca_excluded_clients = []

for client_id, data in client_train_test_data.items():
    n_samples_train = data['X_train'].shape[0]
    if n_samples_train < TARGET_DIM:
        pca_excluded_clients.append((client_id, n_samples_train))
    else:
        clients_after_pca_check[client_id] = data

# 元の辞書を、チェック済みのものに置き換える
client_train_test_data = clients_after_pca_check

print(f"PCA適用可能なクライアント数: {len(client_train_test_data)}")
if pca_excluded_clients:
    print("\n--- PCA適用不可のため除外されたクライアント ---")
    for client_name, count in pca_excluded_clients:
        print(f"クライアント '{client_name}': 訓練サンプル数({count})が目標次元数({TARGET_DIM})未満")
print("-" * 50)
# ▲▲▲▲▲ 修正ここまで ▲▲▲▲▲

# ===================================================================
# Step 5: 各クライアントで中間表現（次元削減後のデータ）を生成
# ===================================================================
print("--- Step 5: 各クライアントで中間表現を生成中... ---")

# --- パラメータ設定 ---
TARGET_DIM = 53 

# 全クライアントの結果を格納する辞書
client_intermediate_representations = {} 

# client_train_test_data の各クライアントについてループ処理
for client_id, data in client_train_test_data.items():
    
    # ---------------------------------------------------------------
    # 1. 各クライアントの「訓練データのみ」を使ってモデルを学習
    # ---------------------------------------------------------------
    # スケーラーとPCAモデルの学習は、必ずそのクライアントの訓練データのみで行います
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(data['X_train'])
    
    pca = PCA(n_components=TARGET_DIM)
    pca.fit(X_train_scaled)
    
    # ---------------------------------------------------------------
    # 2. 学習済みモデルを使い、各データをTARGET_DIM次元に変換
    # ---------------------------------------------------------------
    # a) 訓練データ（学習に使ったデータ自身）
    train_ir = pca.transform(X_train_scaled)
    
    # b) テストデータ (scaler, pca ともに .transform() のみ使用)
    X_test_scaled = scaler.transform(data['X_test'])
    test_ir = pca.transform(X_test_scaled)

    # c) アンカーデータ (scaler, pca ともに .transform() のみ使用)
    X_anchor_scaled = scaler.transform(X_anchor_smote)
    anc_ir = pca.transform(X_anchor_scaled)

    # ---------------------------------------------------------------
    # 3. 結果を辞書に格納
    # ---------------------------------------------------------------
    client_intermediate_representations[client_id] = {
        'train_ir': train_ir,
        'test_ir': test_ir,
        'anc_ir': anc_ir
    }

print("全クライアントの中間表現の生成が完了。")
print("-" * 50)


# ===================================================================
# Step 6: 結果の確認
# ===================================================================
print("--- Step 6: 結果の確認 ---")

# 例として、いずれか1つのクライアントの中間表現の形状を表示
if client_intermediate_representations:
    # 辞書から最初のクライアントのIDを取得
    example_client_id = next(iter(client_intermediate_representations))
    
    print(f"クライアント '{example_client_id.strip()}' の中間表現の形状:")
    print(f"  訓練データ (train_ir): {client_intermediate_representations[example_client_id]['train_ir'].shape}")
    print(f"  テストデータ (test_ir):  {client_intermediate_representations[example_client_id]['test_ir'].shape}")
    print(f"  アンカーデータ (anc_ir): {client_intermediate_representations[example_client_id]['anc_ir'].shape}")

統合表現間の距離でクラスタリングが可能かどうか見る

In [ ]:
INTEGRATED_DIM = 33

all_integrated_reps = {}

# 
anchor_irs_global = {cid: ir_data['anc_ir'] for cid, ir_data in client_intermediate_representations.items()}
G_funcs_global = get_Gfanc_dict(anchor_irs_global, dd=INTEGRATED_DIM)

# ここに client_ids を定義する行を追加します！
client_ids = list(client_intermediate_representations.keys())

global_reps = {}
for client_id in client_ids:
    # ★ 修正点: 変数名を 'client_intermediate_representations' に修正
    ir_data = client_intermediate_representations[client_id]
    g_func = G_funcs_global[client_id]
    
    # ir_data['test_ir_on_c1_data'] のようなキーが存在しないため、
    # 'test_ir' を使用するように修正します。
    train_hat, test_hat, _ = merge_DC(
        ir_data['train_ir'], ir_data['test_ir'], ir_data['anc_ir'], g_func
    )
    
    # 'test_c2' の処理は文脈上不要なため、簡潔にします。
    global_reps[client_id] = {'train': train_hat, 'test': test_hat}

all_integrated_reps['global'] = global_reps

print("\n全データの統合表現化が完了しました。")

In [ ]:
import pandas as pd

# ===================================================================
# Step 10: ある1つの機関の統合表現を行列として表示
# ===================================================================
print("\n--- Step 10: 1つの機関の統合表現を行列形式で表示します ---")

# --- 1. 表示対象のクライアントを選択 ---
# 最初のクライアントを選択します（他のクライアントIDに変更可能）
target_client_id = client_ids[0]
print(f"表示対象のクライアント: '{target_client_id.strip()}'")

# --- 2. 該当クライアントの統合表現（訓練データ）を取得 ---
# all_integrated_repsから'train'の統合表現（Numpy配列）を取得
train_hat = all_integrated_reps['global'][target_client_id]['train']

# --- 3. Numpy配列をPandas DataFrameに変換して表示 ---
# DataFrameに変換することで、見やすい表形式で出力できます
df_hat = pd.DataFrame(train_hat)

print(f"統合表現の行列の形状 (サンプル数, 次元数): {df_hat.shape}")
print("--- 統合表現（先頭5行）---")
# .head()を使うと、行列の先頭部分を表示できます
display(df_hat.head())

In [ ]:
# ===================================================================
#   クラスタリングブロック (Visualization & Verification)
# ===================================================================
# このコードは、メイン実行ブロックの「Step 4 (分割)」完了後、
# 「Step 7 (クラスタごとの学習)」の前に実行することを想定しています。
# 以前の「Step 6 (Global統合表現)」は廃止されたため、
# 統合表現間の距離(EMD)計算は削除し、ラベル分布のTV距離のみを使用します。
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import pandas as pd
# import japanize_matplotlib # 必要に応じて有効化してください
print("\n--- ラベル分布の距離計算(TV距離)と階層的クラスタリング ---")
# 1. データの準備
# client_train_test_data が存在することを前提とします
client_ids = list(client_train_test_data.keys())
num_clients = len(client_ids)
# 全クラス数の取得 (Yall または le から)
# Context に Yall があると仮定、なければ client_train_test_data から推定
try:
    total_num_classes_vis = Yall.nunique()
except NameError:
    # Yallがない場合のフォールバック: 全クライアントの最大ラベル値 + 1
    max_label = 0
    for data in client_train_test_data.values():
        max_label = max(max_label, data['Y_train'].max(), data['Y_test'].max())
    total_num_classes_vis = int(max_label) + 1
    print(f"全クラス数をデータから推定: {total_num_classes_vis}")
# 2. ラベルヒストグラムの作成
print("各クライアントのラベル分布(ヒストグラム)を作成中...")
label_histograms = {}
for cid in client_ids:
    y_train = client_train_test_data[cid]['Y_train']
    
    if y_train.empty:
        counts = np.zeros(total_num_classes_vis)
    else:
        # np.bincount用にint型へ
        counts = np.bincount(y_train.astype(int), minlength=total_num_classes_vis)
    
    # 正規化 (合計=1)
    if counts.sum() == 0:
        label_histograms[cid] = np.zeros(total_num_classes_vis)
    else:
        label_histograms[cid] = counts / counts.sum()
# 3. TV距離 (Total Variation Distance) 行列の計算
# TV Distance = 0.5 * sum(|p - q|)
print("TV距離行列 (Label Distribution) を計算中...")
tv_distance_matrix = np.zeros((num_clients, num_clients))
for i in range(num_clients):
    for j in range(i, num_clients):
        if i == j:
            dist = 0.0
        else:
            p_i = label_histograms[client_ids[i]]
            p_j = label_histograms[client_ids[j]]
            dist = 0.5 * np.sum(np.abs(p_i - p_j))
        
        tv_distance_matrix[i, j] = dist
        tv_distance_matrix[j, i] = dist
print("▼ 作成されたTV距離行列 (一部):")
print(np.round(tv_distance_matrix[:5, :5], 2))
# 4. 階層的クラスタリングの実行
print("\n階層的クラスタリング (Method: Complete Linkage) を実行中...")
# 距離行列をcondensed matrixに変換
condensed_dist = squareform(tv_distance_matrix)
linked = linkage(condensed_dist, method='complete')
# 5. デンドログラムのプロット
plt.figure(figsize=(15, 6))
plt.title(f'Hierarchical Clustering Dendrogram (Total Variation Distance)', fontsize=16)
plt.xlabel('Client ID')
plt.ylabel('Distance (TV)')
# クライアントIDをラベルとして表示
dendrogram(
    linked,
    labels=client_ids,
    leaf_rotation=90.,  # ラベルを90度回転
    leaf_font_size=8.,  # フォントサイズ
)
plt.tight_layout()
plt.show()
# (参考) クラスタリング結果の確認
# t=0.2 は目安です。デンドログラムを見て調整してください。
threshold_t = 0.2
clusters = fcluster(linked, t=threshold_t, criterion='distance')
print(f"\n閾値 t={threshold_t} でのクラスタ数: {len(set(clusters))}")
ids_per_cluster = {}
for cid, clust in zip(client_ids, clusters):
    if clust not in ids_per_cluster: ids_per_cluster[clust] = []
    ids_per_cluster[clust].append(cid)
for c_num in sorted(ids_per_cluster.keys()):
    print(f"Cluster {c_num}: {len(ids_per_cluster[c_num])} clients")

# 最終実験

In [ ]:
# ===================================================================
#                    メインの実行ブロック (Final Refactor v3)
# ===================================================================
# 変更点概要:
# 1. Step 6: TV距離クラスタリング + 【復元】結果表示 & デンドログラム可視化
# 2. Step 7: Global DCモデルの学習・評価 (v2同様)
# 3. Step 8: Cluster-wise DCモデルの学習・評価 (v2同様)
# 4. 集計: Local, Global DC, Cluster DC (re-calc G) の3つ

import matplotlib.pyplot as plt

# --- 実験の全体設定 ---
N_TRIALS = 1  # 試行回数
all_client_centric_results = []
all_local_results = []

# 事前に定義が必要な変数 (Contextから取得想定)
try:
    total_num_classes_run = Yall.nunique()
except NameError:
    print("Warning: 'Yall' not found. Ensure data is loaded.")
    total_num_classes_run = 7 

INTEGRATED_DIM_LEARNING = 35    # 学習用の統合表現次元数

# --- 複数回試行のループ ---
for i in range(N_TRIALS):
    base_seed = i * 10
    print("\n" + "="*80)
    print(f"TRIAL {i+1}/{N_TRIALS} (Base Seed: {base_seed})")
    print("="*80)
    
    # シード固定
    np.random.seed(base_seed)
    random.seed(base_seed)
    
    # -------------------------------------------------------------------
    # Step 2: 公開データの抽出
    # -------------------------------------------------------------------
    X_others, X_public, Y_others, Y_public = train_test_split(
        Xall, 
        Yall, 
        test_size=100,
        random_state=base_seed,
        stratify=Yall
    )
    
    # -------------------------------------------------------------------
    # Step 3: 各クライアントへのデータ分割
    # -------------------------------------------------------------------
    print("\n--- Step 3: クライアントデータ生成 ---")
    NUM_CLIENTS = 100
    K_LABELS_PER_CLIENT = 3
 
    np.random.seed(base_seed + 1)
    random.seed(base_seed + 1)
 
    all_labels = sorted(Y_others.unique())
    indices_per_label = {label: Y_others.index[Y_others == label].tolist() for label in all_labels}
    client_label_assignments = {idx: [] for idx in range(NUM_CLIENTS)}
    available_labels = list(all_labels)

    # クライアントへのラベル割当
    for client_idx in range(NUM_CLIENTS):
        assigned_labels = np.random.choice(available_labels, K_LABELS_PER_CLIENT, replace=False)
        client_label_assignments[client_idx] = list(assigned_labels)
 
    label_to_clients_map = {label: [] for label in all_labels}
    for client_id, labels in client_label_assignments.items():
        for label in labels:
            label_to_clients_map[label].append(client_id)
 
    client_indices = {idx: [] for idx in range(NUM_CLIENTS)}
    for label, indices_for_this_label in indices_per_label.items():
        owning_clients = label_to_clients_map.get(label, [])
        if not owning_clients: continue
        shuffled_indices = indices_for_this_label.copy()
        random.shuffle(shuffled_indices)
        index_chunks = np.array_split(shuffled_indices, len(owning_clients))
        for client_id, chunk in zip(owning_clients, index_chunks):
            if chunk.size > 0:
                client_indices[client_id].extend(chunk.tolist())
 
    client_data = {}
    for client_idx in range(NUM_CLIENTS):
        client_name = f"client_{client_idx}"
        indices = client_indices.get(client_idx, [])
        if not indices:
            client_data[client_name] = {'X': pd.DataFrame(), 'Y': pd.Series(dtype='int')}
        else:
            client_data[client_name] = {'X': X_others.loc[indices], 'Y': Y_others.loc[indices]}

    # -------------------------------------------------------------------
    # Step 3.5 & 4: フィルタリング と Train/Test 分割
    # -------------------------------------------------------------------
    filtered_client_data = {}
    for client_name, data in client_data.items():
        if data['Y'].nunique() == K_LABELS_PER_CLIENT:
            filtered_client_data[client_name] = data
    client_data = filtered_client_data
    
    print("\n--- Step 4: Train/Test 分割 ---")
    MIN_SAMPLES_THRESHOLD = 5
    TEST_SPLIT_RATIO = 0.2
    client_train_test_data = {}
    
    for client_id, data in client_data.items():
        X_c, Y_c = data['X'], data['Y']
        if len(X_c) < MIN_SAMPLES_THRESHOLD: continue
        try:
            X_train, X_test, Y_train, Y_test = train_test_split(
                X_c, Y_c, test_size=TEST_SPLIT_RATIO, random_state=base_seed+2, stratify=None
            )
            if X_train.empty: continue
            
            client_train_test_data[client_id] = {
                'X_train': X_train, 'Y_train': Y_train,
                'X_test': X_test, 'Y_test': Y_test
            }
        except ValueError:
            continue
            
    print(f"有効クライアント数: {len(client_train_test_data)}")

    # -------------------------------------------------------------------
    # Step 5: ローカル表現(PCA)の生成 & Localモデル評価
    # -------------------------------------------------------------------
    print("\n--- Step 5: ローカル表現(PCA)生成 & Localモデル学習 ---")
    TARGET_DIM = 53
    
    clients_after_pca = {}
    for cid, data in client_train_test_data.items():
        if data['X_train'].shape[0] >= TARGET_DIM:
            clients_after_pca[cid] = data
    client_train_test_data = clients_after_pca

    X_anchor_smote = make_anc_smote(X_public, r=1000, k=25, a=1.5, rs=base_seed+3)
    
    client_intermediate_representations = {}
    
    for cid, data in client_train_test_data.items():
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(data['X_train'])
        
        pca = PCA(n_components=TARGET_DIM, random_state=base_seed+4)
        pca.fit(X_train_scaled)
        
        if data['X_test'].empty:
            X_test_scaled = np.array([]).reshape(0, X_train_scaled.shape[1])
            test_ir = np.array([]).reshape(0, TARGET_DIM)
        else:
            X_test_scaled = scaler.transform(data['X_test'])
            test_ir = pca.transform(X_test_scaled)
            
        client_intermediate_representations[cid] = {
            'train_ir': pca.transform(X_train_scaled),
            'test_ir': test_ir,
            'anc_ir': pca.transform(scaler.transform(X_anchor_smote))
        }

        # Localモデル評価
        metrics, _ = train_and_evaluate_pytorch(
            X_train_scaled, data['Y_train'].values,
            X_test_scaled, data['Y_test'].values,
            total_num_classes=total_num_classes_run, seed=base_seed+5
        )
        all_local_results.append({
            'trial': i, 'client_id': cid, 'Accuracy': metrics['Accuracy']
        })

    # ===================================================================
    # Step 6: クラスタリング (ラベル分布 TV距離)
    # ===================================================================
    print("\n--- Step 6: ラベル分布に基づくクラスタリング (TV Distance) ---")
    from scipy.spatial.distance import squareform
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    
    client_ids = list(client_intermediate_representations.keys())
    num_clients = len(client_ids)
    
    label_histograms = {}
    for cid in client_ids:
        y_tr = client_train_test_data[cid]['Y_train']
        counts = np.bincount(y_tr.astype(int), minlength=total_num_classes_run)
        label_histograms[cid] = counts / counts.sum() if counts.sum() > 0 else np.zeros(total_num_classes_run)

    dist_matrix = np.zeros((num_clients, num_clients))
    for j in range(num_clients):
        for k in range(j+1, num_clients):
            p_j = label_histograms[client_ids[j]]
            p_k = label_histograms[client_ids[k]]
            dist = 0.5 * np.sum(np.abs(p_j - p_k)) # Total Variation
            dist_matrix[j, k] = dist_matrix[k, j] = dist
            
    condensed_dist = squareform(dist_matrix)
    linked = linkage(condensed_dist, method='complete')
    
    clusters = fcluster(linked, t=0.2, criterion='distance')
    
    clusters_dict = {}
    client_id_to_cluster = {}
    for idx, cid in enumerate(client_ids):
        cluster_id = clusters[idx]
        if cluster_id not in clusters_dict:
            clusters_dict[cluster_id] = []
        clusters_dict[cluster_id].append(cid)
        client_id_to_cluster[cid] = cluster_id
        
    print(f"クラスタ数: {len(clusters_dict)}")

    # --------- 【可視化復元】 クラスタリング結果の表示 ---------
    print("\n--- クラスタリング結果 ---")
    for cluster_num, members in sorted(clusters_dict.items()):
        cleaned_members = [str(m) for m in members]
        print(f"クラスタ {cluster_num}: {cleaned_members}")

    print(f"TRIAL {i+1}: デンドログラムをプロット中...")
    cleaned_labels = [str(cid) for cid in client_ids]
    plt.figure(figsize=(18, 8))

    plt.title(
        f'TRIAL {i+1} / Seed {base_seed} - Hierarchical Clustering (TV Distance)',
        fontsize=16
    )
    dendrogram(linked, labels=cleaned_labels, orientation='top', leaf_rotation=90)
    plt.xlabel('Client ID', fontsize=12)
    plt.ylabel('Distance (TV Distance)', fontsize=12)

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    # -------------------------------------------------------------

    # ===================================================================
    # Step 7: Global統合表現の作成 & Global DCモデル (復元)
    # ===================================================================
    print("\n--- Step 7: Global DCモデルの学習 ---")
    
    # 1. Global Anchor & G関数
    anchor_irs_global = {cid: ir_data['anc_ir'] for cid, ir_data in client_intermediate_representations.items()}
    G_funcs_global = get_Gfanc_dict(anchor_irs_global, dd=INTEGRATED_DIM_LEARNING)
    
    # 2. 統合表現の作成
    global_train_list, global_Y_train_list = [], []
    global_test_map = {}
    
    for cid, ir_data in client_intermediate_representations.items():
        train_hat, test_hat, _ = merge_DC(
            ir_data['train_ir'], ir_data['test_ir'], ir_data['anc_ir'], G_funcs_global[cid]
        )
        global_train_list.append(train_hat)
        global_Y_train_list.append(client_train_test_data[cid]['Y_train'])
        global_test_map[cid] = test_hat
        
    # 3. スタック & Global標準化
    X_train_global_stacked = np.vstack(global_train_list)
    scaler_global = StandardScaler()
    X_train_global_std = scaler_global.fit_transform(X_train_global_stacked)
    Y_train_global_std = pd.concat(global_Y_train_list)
    
    # 4. Global DCモデル学習
    _, model_global_dc = train_and_evaluate_pytorch(
        X_train_global_std, Y_train_global_std.values,
        np.array([]), np.array([]), 
        total_num_classes=total_num_classes_run, seed=base_seed+5
    )
    
    # 5. Global DCモデル評価 (各クライアント)
    global_dc_scores = {}
    for cid in client_ids:
        X_test_unscaled = global_test_map[cid]
        Y_test = client_train_test_data[cid]['Y_test'].values
        if len(X_test_unscaled) > 0:
            X_test_std = scaler_global.transform(X_test_unscaled)
            acc = evaluate_pytorch_model(model_global_dc, X_test_std, Y_test)
        else:
            acc = np.nan
        global_dc_scores[cid] = acc

    # ===================================================================
    # Step 8: クラスタ単位でのモデル学習 (DC-CFL Cluster-wise)
    # ===================================================================
    print("\n--- Step 8: Cluster-wise DCモデルの学習 (re-calc G) ---")
    
    for cluster_num, members in clusters_dict.items():
        # メンバー数が少ない場合はLocalスコアを使用
        if len(members) < 2:
            print(f"Cluster {cluster_num}: メンバー数({len(members)})不足 -> ClusterスコアはLocalで代用")
            for cid in members:
                # Localを取得
                local_acc = next((r['Accuracy'] for r in all_local_results 
                                  if r['client_id'] == cid and r['trial'] == i), np.nan)
                # Global DCも取得
                gdc_acc = global_dc_scores.get(cid, np.nan)
                
                all_client_centric_results.append({
                    'trial': i, 'client_id': cid, 'cluster': cluster_num,
                    'DC_Global': gdc_acc,
                    'DC_Cluster_Score': local_acc
                })
            continue

        print(f"Cluster {cluster_num} (Metrics: TV-Dist): 学習中 (Members: {len(members)})")
        
        # 1. クラスタ内アンカー
        cluster_anchor_irs = {cid: client_intermediate_representations[cid]['anc_ir'] for cid in members}
        
        # 2. クラスタ内 G関数
        G_funcs_cluster = get_Gfanc_dict(cluster_anchor_irs, dd=INTEGRATED_DIM_LEARNING)
        
        # 3. 統合表現の作成
        X_train_list, Y_train_list = [], []
        test_ir_map_cluster = {} 
        
        for cid in members:
            ir_data = client_intermediate_representations[cid]
            train_hat, test_hat, _ = merge_DC(
                ir_data['train_ir'], ir_data['test_ir'], ir_data['anc_ir'], G_funcs_cluster[cid]
            )
            X_train_list.append(train_hat)
            Y_train_list.append(client_train_test_data[cid]['Y_train'])
            test_ir_map_cluster[cid] = test_hat
            
        # 4. スタック & クラスタ内標準化
        X_train_stacked = np.vstack(X_train_list)
        scaler_cluster = StandardScaler() 
        X_train_stacked_std = scaler_cluster.fit_transform(X_train_stacked)
        Y_train_stacked = pd.concat(Y_train_list)
        
        # 5. Cluster Model学習
        _, model_cluster = train_and_evaluate_pytorch(
            X_train_stacked_std, Y_train_stacked.values,
            np.array([]), np.array([]),
            total_num_classes=total_num_classes_run, seed=base_seed+6
        )
        
        # 6. Cluster Model評価 & 結果保存
        for cid in members:
            X_test_unscaled = test_ir_map_cluster[cid]
            Y_test = client_train_test_data[cid]['Y_test'].values
            
            if len(X_test_unscaled) > 0:
                X_test_std = scaler_cluster.transform(X_test_unscaled)
                acc = evaluate_pytorch_model(model_cluster, X_test_std, Y_test)
            else:
                acc = np.nan
            
            # Globalスコアも併記
            gdc_acc = global_dc_scores.get(cid, np.nan)
            
            all_client_centric_results.append({
                'trial': i, 'client_id': cid, 'cluster': cluster_num,
                'DC_Global': gdc_acc,
                'DC_Cluster_Score': acc
            })

    print(f"TRIAL {i+1} 完了。")

# --- 結果集計 ---
print("\n" + "="*80)
print("集計結果 Summary")
print("="*80)
df_local = pd.DataFrame(all_local_results)
df_dc = pd.DataFrame(all_client_centric_results)

if not df_dc.empty and not df_local.empty:
    merged = pd.merge(df_dc, df_local, on=['trial', 'client_id'], suffixes=('_dc', '_local'))
    
    # 試行ごとの平均
    print("\n--- Per-Trial Mean Accuracy ---")
    print(merged.groupby('trial')[['Accuracy', 'DC_Global', 'DC_Cluster_Score']].mean().round(4))
    
    # 全体平均
    print("\n--- Overall Mean Accuracy ---")
    print(merged[['Accuracy', 'DC_Global', 'DC_Cluster_Score']].mean().round(4))
else:
    print("有効な結果が得られませんでした。")